# Week 7: imbalance strategies for the 16-base-code ECG task

**Purpose.** Compare two predeclared training-loss changes against both a matched unweighted replication and the untouched first multilabel baseline on exactly the same patient-level training and validation records, waveform pipeline, ResNet-1D architecture, optimizer settings, seed, 16 target mapping, checkpoint rule, and validation records. This is exploratory validation work, not test evidence or clinical validation.

**Candidates.** (1) Positive-weighted BCE with `pos_weight = min((N_train-N_positive)/N_positive, 20)` per target, computed from training labels only. PyTorch documents positive weights as a way to trade precision for recall: [BCEWithLogitsLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html). (2) Asymmetric loss (ASL) for multilabel imbalance, with `gamma_pos=0`, `gamma_neg=4`, negative probability clipping `0.05`, following the [original ICCV paper](https://openaccess.thecvf.com/content/ICCV2021/papers/Ridnik_Asymmetric_Loss_for_Multi-Label_Classification_ICCV_2021_paper.pdf) and [authors' implementation](https://github.com/Alibaba-MIIL/ASL). This is an experimental transfer from visual multi-label classification to ECG, not an established method for this dataset. Parameters are fixed before validation inspection.

**Comparability.** The matched unweighted replication and both candidates are randomly initialized and train from scratch for at most eight epochs with patience three. All checkpoints, including the preserved unweighted baseline, were selected by minimum **unweighted validation BCE**. The common primary report uses fixed threshold 0.5; a fixed 0.2 threshold is shown as a sensitivity analysis for all three models equally, never optimized on validation. Ranking metrics and positive-patient support accompany threshold metrics. No candidate is declared better from recall alone. Small differences on this repeatedly used validation set are exploratory.

**Label and test boundaries.** The original 19 exact strings remain the source of truth. The existing mapping manifest determines the 16 outputs. A selected code's absence is an operational zero, not confirmed clinical absence. Only patient IDs are accessed to reconstruct the reserved split; held-out diagnosis fields and waveforms are skipped. No Week 4–6 or first Week 7 artifacts are overwritten.

In [1]:
from pathlib import Path
import csv, random, re, json, os, sys, platform, importlib.util
from collections import Counter
import numpy as np
import pandas as pd
import torch, wfdb
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Data/AttributesDictionary.csv').is_file())
OUT=ROOT/'new work/Week7/experimental/imbalance_strategies'
BASE=ROOT/'new work/Week7/experimental/multilabel_base16'
assert OUT.is_dir() and BASE.is_dir()
assert Path(sys.executable).resolve()==(ROOT/'venv/bin/python').resolve(), 'Use existing project venv kernel'
assert torch.cuda.is_available(), 'CUDA unavailable in actual notebook kernel; stop before training'
DEVICE=torch.device('cuda:0')
assert (torch.ones(1,device=DEVICE)*2).item()==2
print('Python',sys.executable,platform.python_version(),'Torch',torch.__version__,'CUDA',torch.cuda.get_device_name(0))
SEED=42;FS=500;N_SAMPLES=5000;BATCH_SIZE=32;MAX_EPOCHS=8;PATIENCE=3;LR=3e-4;WEIGHT_DECAY=1e-4
LEADS=['I','II','III','AVR','AVL','AVF','V1','V2','V3','V4','V5','V6'];LI={x:i for i,x in enumerate(LEADS)}
MAP=json.loads((BASE/'target_mapping.json').read_text()); exact_codes=MAP['exact_codes'];base_codes=MAP['base_codes_in_model_order'];base_of=MAP['exact_to_base']
assert len(exact_codes)==19 and len(base_codes)==16 and len(set(base_codes))==16
base_index={x:i for i,x in enumerate(base_codes)};exact_set=set(exact_codes)
torch.set_num_threads(min(4,os.cpu_count() or 1))

Python /home/souhit/Documents/Research/Pediatric_ECG/venv/bin/python 3.13.5 Torch 2.6.0+cu124 CUDA NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
META=ROOT/'Data/AttributesDictionary.csv'
with META.open(newline='') as f:
 reader=csv.reader(f);h=next(reader);pi=h.index('Patient_ID');patient_ids=sorted({r[pi] for r in reader})
random.Random(SEED).shuffle(patient_ids)
a=int(.70*len(patient_ids));b=int(.15*len(patient_ids))
train_ids=set(patient_ids[:a]);val_ids=set(patient_ids[a:a+b]);test_ids=set(patient_ids[a+b:])
assert (len(train_ids),len(val_ids),len(test_ids))==(8150,1746,1747)
def tokens(x):return [v.strip().strip("'\"").strip() for v in x.split(';') if v.strip()]
train_rows=[];val_rows=[]
with META.open(newline='') as f:
 for r in csv.DictReader(f):
  pid=r['Patient_ID']
  if pid in test_ids:continue # no held-out ICD/age/lead/signal access
  assert pid in train_ids or pid in val_ids
  selected=set(tokens(r['ICD-10 code']))&exact_set
  y=np.zeros(16,dtype=np.float32)
  for x in selected:y[base_index[base_of[x]]]=1
  item={'patient_id':pid,'ecg_id':r['ECG_ID'],'filename':r['Filename'],'sampling_point':int(r['Sampling_point']),
        'age_years':int(r['Age'][:-1])/365.25,'lead_count':int(r['Lead']),'target':y}
  (train_rows if pid in train_ids else val_rows).append(item)
assert (len(train_rows),len(val_rows))==(9943,2169)
Y_train=np.stack([r['target'] for r in train_rows]);Y_val=np.stack([r['target'] for r in val_rows])
assert Y_train.shape==(9943,16) and Y_val.shape==(2169,16)
prior=pd.read_csv(BASE/'validation_predictions.csv',dtype={'patient_id':str,'ecg_id':str})
assert prior.patient_id.tolist()==[r['patient_id'] for r in val_rows]
assert prior.ecg_id.tolist()==[r['ecg_id'] for r in val_rows]
assert np.array_equal(prior[[f'target::{x}' for x in base_codes]].to_numpy(dtype=np.float32),Y_val)
counts=Y_train.sum(0);pos_weight=np.minimum((len(Y_train)-counts)/counts,20).astype(np.float32)
assert (counts>0).all()
print('Exact baseline validation rows and target matrix reproduced; positive weights from training only:')
display(pd.DataFrame({'code':base_codes,'train_positive_records':counts.astype(int),'pos_weight':pos_weight}).round(3))

Exact baseline validation rows and target matrix reproduced; positive weights from training only:


,code,train_positive_records,pos_weight
0,I40.0,296,20.000
1,I40.9,39,20.000
2,I51.4,106,20.000
3,I42.0,44,20.000
4,I42.2,18,20.000
5,I42.9,17,20.000
6,Q24.8,14,20.000
7,M30.3,134,20.000
8,Q21.0,723,12.752
9,Q21.1,1158,7.586


In [3]:
RECORDS=ROOT/'Child_ECG/Child_ecg'
class ECGDataset(Dataset):
 def __init__(self,rows):self.rows=rows
 def __len__(self):return len(self.rows)
 def __getitem__(self,i):
  r=self.rows[i];signal,fields=wfdb.rdsamp(str(RECORDS/r['filename']),sampto=min(N_SAMPLES,r['sampling_point']))
  assert fields['fs']==FS and np.isfinite(signal).all()
  x=np.zeros((12,N_SAMPLES),dtype=np.float32)
  for j,lead in enumerate(fields['sig_name']):
   k=LI[lead.upper()];v=signal[:,j].astype(np.float32);sd=v.std()
   x[k,:len(v)]=(v-v.mean())/sd if sd>1e-8 else 0
  return torch.from_numpy(x),torch.from_numpy(r['target'])
train_data=ECGDataset(train_rows);val_data=ECGDataset(val_rows)
val_loader=DataLoader(val_data,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)
spec=importlib.util.spec_from_file_location('week5_resnet',ROOT/'NoteBooks/Week5/models.py')
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module);ResNet1D=module.ResNet1D
common_bce=torch.nn.BCEWithLogitsLoss()
class AsymmetricLoss(torch.nn.Module):
 def __init__(self,gamma_pos=0,gamma_neg=4,clip=.05,eps=1e-8):
  super().__init__();self.gp=gamma_pos;self.gn=gamma_neg;self.clip=clip;self.eps=eps
 def forward(self,logits,y):
  pp=torch.sigmoid(logits);pn=1-pp
  pn=(pn+self.clip).clamp(max=1)
  log_likelihood=y*torch.log(pp.clamp(min=self.eps))+(1-y)*torch.log(pn.clamp(min=self.eps))
  with torch.no_grad():
   pt=pp*y+pn*(1-y);gamma=self.gp*y+self.gn*(1-y)
   focal=(1-pt).pow(gamma)
  return -(log_likelihood*focal).mean()
def seed_all():
 random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
 torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
def train_loader():
 g=torch.Generator().manual_seed(SEED)
 return DataLoader(train_data,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True,generator=g)
def validate(model):
 model.eval();tot=0.;ys=[];ps=[]
 with torch.no_grad():
  for x,y in val_loader:
   x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
   logits=model(x);tot+=common_bce(logits,y).item()*len(y)
   ys.append(y.cpu().numpy());ps.append(torch.sigmoid(logits).cpu().numpy())
 return tot/len(val_data),np.concatenate(ys),np.concatenate(ps)
config={'candidates':{'weighted_bce':{'pos_weight_rule':'min(negative/positive training records,20)','values':dict(zip(base_codes,[float(x) for x in pos_weight]))},
                      'asymmetric_loss':{'gamma_pos':0,'gamma_neg':4,'negative_probability_clip':.05}},
 'architecture':'NoteBooks/Week5/models.py:ResNet1D(num_classes=16), random init',
 'baseline':'new work/Week7/experimental/multilabel_base16/resnet1d_base16_best.pth',
 'seed':SEED,'split':'sorted patient IDs, seed42, floor 70/15/15','input_samples':N_SAMPLES,'lead_order':LEADS,
 'batch_size':BATCH_SIZE,'optimizer':'Adam','learning_rate':LR,'weight_decay':WEIGHT_DECAY,'max_epochs':MAX_EPOCHS,'patience':PATIENCE,
 'checkpoint_rule':'minimum common unweighted validation BCE; delta 1e-5','thresholds_reported':[.5,.2],
 'test_labels_or_waveforms_accessed':False,'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0)}
(OUT/'config.json').write_text(json.dumps(config,indent=2)+'\n')

1452

## Matched unweighted replication and two loss strategies

All three new runs get identical random initialization and shuffling seed. Each training objective differs, while model selection uses the same unweighted validation BCE as the untouched baseline. Every candidate retains its checkpoint and history.

In [4]:
def run_candidate(name,criterion):
 seed_all();model=ResNet1D(num_classes=16).to(DEVICE)
 optimizer=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
 loader=train_loader();history=[];best=float('inf');best_epoch=None;stale=0
 path=OUT/f'{name}_best.pth'
 for epoch in range(1,MAX_EPOCHS+1):
  model.train();loss_sum=0.;bce_sum=0.
  for x,y in loader:
   x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
   logits=model(x);loss=criterion(logits,y)
   with torch.no_grad():bce_sum+=common_bce(logits,y).item()*len(y)
   optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()
   loss_sum+=loss.item()*len(y)
  val_bce,vy,vp=validate(model)
  assert np.array_equal(vy,Y_val)
  history.append({'epoch':epoch,'training_objective':loss_sum/len(train_data),'training_unweighted_bce':bce_sum/len(train_data),'validation_unweighted_bce':val_bce})
  pd.DataFrame(history).to_csv(OUT/f'{name}_history.csv',index=False)
  if val_bce<best-1e-5:
   best=val_bce;best_epoch=epoch;stale=0
   torch.save({'model_state_dict':model.state_dict(),'architecture':'ResNet1D','num_classes':16,
               'base_codes_in_model_order':base_codes,'exact_to_base':base_of,'original_exact_codes':exact_codes,
               'strategy':name,'epoch':epoch,'validation_unweighted_bce':val_bce,'seed':SEED,
               'input_samples':N_SAMPLES,'lead_order':LEADS,'selection':'minimum common unweighted validation BCE'},path)
  else:stale+=1
  print(f'{name} epoch {epoch}: train objective {loss_sum/len(train_data):.5f}, val BCE {val_bce:.5f}, best {best_epoch}',flush=True)
  if stale>=PATIENCE:break
 saved=torch.load(path,map_location=DEVICE,weights_only=False)
 model.load_state_dict(saved['model_state_dict'])
 val_bce,vy,vp=validate(model)
 assert abs(val_bce-saved['validation_unweighted_bce'])<1e-6
 return {'epoch':best_epoch,'bce':val_bce,'y':vy,'p':vp,'checkpoint':path.name}
runs={'unweighted_replicate':run_candidate('unweighted_replicate',common_bce)}
print('Matched unweighted replication complete. Starting weighted BCE.',flush=True)
weighted_criterion=torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight,device=DEVICE))
runs['weighted_bce']=run_candidate('weighted_bce',weighted_criterion)
print('Weighted BCE complete. Starting ASL.',flush=True)
runs['asymmetric_loss']=run_candidate('asymmetric_loss',AsymmetricLoss())
print('Unweighted replication and both candidates complete.',flush=True)

unweighted_replicate epoch 1: train objective 0.10871, val BCE 0.06824, best 1


unweighted_replicate epoch 2: train objective 0.06671, val BCE 0.06537, best 2


unweighted_replicate epoch 3: train objective 0.06384, val BCE 0.06291, best 3


unweighted_replicate epoch 4: train objective 0.06232, val BCE 0.06251, best 4


unweighted_replicate epoch 5: train objective 0.06055, val BCE 0.06355, best 4


unweighted_replicate epoch 6: train objective 0.05960, val BCE 0.06104, best 6


... (13 outputs left out)

asymmetric_loss epoch 3: train objective 0.02069, val BCE 0.48499, best 2


asymmetric_loss epoch 4: train objective 0.02043, val BCE 0.44259, best 2


asymmetric_loss epoch 5: train objective 0.02009, val BCE 0.43511, best 2


Unweighted replication and both candidates complete.


## Common validation comparison

The saved first-baseline probability table is joined by record order and IDs, then all models are scored by the same implementation. AUROC and AP are reported only for targets with both classes. Zero-positive targets are listed with undefined metrics; targets with fewer than ten positive validation patients remain exploratory. Fixed threshold 0.2 is a common sensitivity analysis, not a tuned operating point.

In [5]:
baseline_p=prior[[f'probability::{x}' for x in base_codes]].to_numpy(dtype=float)
runs={'unweighted_baseline':{'epoch':int(json.loads((BASE/'aggregate_validation.json').read_text())['selected_epoch']),
                              'bce':float(json.loads((BASE/'aggregate_validation.json').read_text())['validation_bce']),
                              'y':Y_val,'p':baseline_p,'checkpoint':'prior Week7 baseline'},**runs}
def metrics(y,p,threshold):
 pred=p>=threshold;flat_y=y.ravel().astype(int);flat_p=p.ravel();flat_pred=pred.ravel()
 tp=int(((flat_y==1)&flat_pred).sum());tn=int(((flat_y==0)&(~flat_pred)).sum())
 fp=int(((flat_y==0)&flat_pred).sum());fn=int(((flat_y==1)&(~flat_pred)).sum())
 aps=[];aucs=[];per=[]
 for j,c in enumerate(base_codes):
  yy=y[:,j].astype(int);pp=p[:,j];zz=pred[:,j]
  a=int(((yy==1)&zz).sum());b=int(((yy==0)&zz).sum());d=int(((yy==1)&(~zz)).sum());e=int(((yy==0)&(~zz)).sum())
  positive_patients=len({r['patient_id'] for r,positive in zip(val_rows,yy) if positive})
  ap=average_precision_score(yy,pp) if 0<yy.sum()<len(yy) else np.nan
  auc=roc_auc_score(yy,pp) if 0<yy.sum()<len(yy) else np.nan
  aps.append(ap);aucs.append(auc)
  per.append({'code':c,'positive_records':int(yy.sum()),'positive_patients':positive_patients,
              'tn':e,'fp':b,'fn':d,'tp':a,'sensitivity':a/(a+d) if a+d else np.nan,
              'specificity':e/(e+b) if e+b else np.nan,'precision':a/(a+b) if a+b else np.nan,
              'f1':2*a/(2*a+b+d) if a+d else np.nan,'auroc':auc,'average_precision':ap})
 return {'micro_tp':tp,'micro_fp':fp,'micro_fn':fn,'micro_tn':tn,'micro_sensitivity':tp/(tp+fn),
         'micro_specificity':tn/(tn+fp),'micro_precision':tp/(tp+fp) if tp+fp else np.nan,
         'micro_f1':2*tp/(2*tp+fp+fn),'micro_auroc':roc_auc_score(flat_y,flat_p),
         'micro_average_precision':average_precision_score(flat_y,flat_p),
         'macro_auroc':np.nanmean(aucs),'macro_average_precision':np.nanmean(aps),
         'macro_sensitivity':np.nanmean([z['sensitivity'] for z in per]),
         'macro_f1':np.nanmean([z['f1'] for z in per]),'subset_accuracy':float(np.mean(np.all(pred==y,axis=1))),
         'defined_ranking_targets':int(np.isfinite(aps).sum())},pd.DataFrame(per)
comparison=[];per_target=[]
for name,run in runs.items():
 assert np.array_equal(run['y'],Y_val)
 for threshold in [.5,.2]:
  m,table=metrics(Y_val,run['p'],threshold)
  comparison.append({'model':name,'threshold':threshold,'epoch':run['epoch'],'validation_unweighted_bce':run['bce'],**m})
  table.insert(0,'model',name);table.insert(1,'threshold',threshold);per_target.append(table)
comparison=pd.DataFrame(comparison);per_target=pd.concat(per_target,ignore_index=True)
comparison.to_csv(OUT/'validation_comparison.csv',index=False)
per_target.to_csv(OUT/'per_target_comparison.csv',index=False)
display(comparison.round(4))
for name,run in runs.items():
 if name=='unweighted_baseline':continue
 d=prior[['patient_id','ecg_id']].copy()
 for j,c in enumerate(base_codes):d[f'probability::{c}']=run['p'][:,j]
 d.to_csv(OUT/f'{name}_validation_predictions.csv',index=False)
print('All model outputs evaluated on identical validation rows')

,model,threshold,epoch,validation_unweighted_bce,micro_tp,micro_fp,micro_fn,micro_tn,micro_sensitivity,micro_specificity,micro_precision,micro_f1,micro_auroc,micro_average_precision,macro_auroc,macro_average_precision,macro_sensitivity,macro_f1,subset_accuracy,defined_ranking_targets
0,unweighted_baseline,0.5,8,0.0611,147,67,524,33966,0.2191,0.9980,0.6869,0.3322,0.9131,0.3999,0.8283,0.1639,0.0752,0.1084,0.7621,14
1,unweighted_baseline,0.2,8,0.0611,297,416,374,33617,0.4426,0.9878,0.4165,0.4292,0.9131,0.3999,0.8283,0.1639,0.1686,0.1505,0.7211,14
2,unweighted_replicate,0.5,6,0.0610,125,49,546,33984,0.1863,0.9986,0.7184,0.2959,0.9133,0.3990,0.8375,0.1819,0.0460,0.0653,0.7616,14
3,unweighted_replicate,0.2,6,0.0610,298,381,373,33652,0.4441,0.9888,0.4389,0.4415,0.9133,0.3990,0.8375,0.1819,0.1750,0.1716,0.7192,14
4,weighted_bce,0.5,4,0.1597,402,1845,269,32188,0.5991,0.9458,0.1789,0.2755,0.9126,0.2850,0.8294,0.1441,0.3146,0.1289,0.4338,14
5,weighted_bce,0.2,4,0.1597,560,6072,111,27961,0.8346,0.8216,0.0844,0.1534,0.9126,0.2850,0.8294,0.1441,0.5719,0.0955,0.0456,14
6,asymmetric_loss,0.5,2,0.4244,495,4288,176,29745,0.7377,0.8740,0.1035,0.1815,0.8929,0.2876,0.7913,0.1370,0.3722,0.0790,0.0383,14
7,asymmetric_loss,0.2,2,0.4244,669,27411,2,6622,0.9970,0.1946,0.0238,0.0465,0.8929,0.2876,0.7913,0.1370,0.9906,0.0440,0.0000,14


All model outputs evaluated on identical validation rows


## Interpretation

This notebook compares two loss changes under identical architecture and data. Any observed advantage is exploratory because validation has already been used for checkpoint selection and earlier design decisions. Candidate objectives can change score calibration, so fixed-threshold changes need not reflect better ranking. Per-target AP is prevalence-dependent; per-target values for one or a few positive patients are unstable. Repeated ECGs from a patient are correlated. The dataset's inpatient codes may be incomplete and are not verified ECG-time disease states. No test data are used.

## Observed results and decision

At the common 0.5 threshold on 2,169 validation records, the untouched unweighted baseline has **TP 147 / FP 67 / FN 524**, micro sensitivity **0.219**, F1 **0.332**, micro AP **0.400**, and macro AP **0.164**. The matched unweighted replication has **TP 125 / FP 49 / FN 546**, F1 **0.296**, micro AP **0.399**, and macro AP **0.182**. This run-to-run difference is a reminder not to attribute every metric change to a new strategy.

Positive-weighted BCE raises TP to **402** but FP to **1,845**; precision falls to **0.179**, F1 to **0.276**, micro AP to **0.285**, and macro AP to **0.144**. Asymmetric loss raises TP to **495** and FP to **4,288**; precision is **0.103**, F1 **0.182**, micro AP **0.288**, and macro AP **0.137**. Their higher recall at this threshold is not a broad improvement. A common threshold of 0.2 further increases recall while causing very large false-positive counts. Because these losses alter score scale, fixed-threshold recall changes alone are not evidence of better discrimination.

**Decision:** Retain the original unweighted baseline checkpoint. The weighted and asymmetric checkpoints and complete histories are preserved as negative experimental results. Do not promote either candidate. See `validation_comparison.csv` and `per_target_comparison.csv` for all metrics and per-code counts.